In [87]:
# @launchit.collected

In [86]:
project_root_dir = !git rev-parse --show-toplevel
project_root_dir = project_root_dir[0]

import os
import sys
import json
import re
from collections import namedtuple # @launchit.collect
from enum import IntEnum, auto # @launchit.collect
import time

# @launchit.disable
sys.path.append(os.path.join(project_root_dir, 'lib'))
import launchit

In [88]:
# @launchit.disable
# @launchit.collect
class Command(IntEnum):
    COLLECT = auto()
    COLLECTED = auto()
    DISABLE = auto()

In [89]:
# launchit_t0 = time.time()

In [90]:
# launchit_interval = time.time() - launchit_t0

# if launchit_interval > 0.05:
#     launchit.launchit(os.environ['JPY_SESSION_NAME']) # @launchit.disable
# else:
#     print('Skip launchit due to mass "Run Cells"')

In [117]:
# @launchit.disable
# @launchit.collect
MODEL_INSTANCE_INFO = namedtuple('ModelInstanceInfo', 'group_uri, name, version, main_asset_fname')(
    group_uri='${MODEL_GROUP_URI}',
    name='${MODEL_NAME}',
    version='${MODEL_VERSION}',
    main_asset_fname='{NOTEBOOK_FNAME}'
)
# @launchit.stop
# @launchit.stop

# @launchit.disable
MODEL_INSTANCE_INFO = MODEL_INSTANCE_INFO._replace(group_uri='group_uri')
MODEL_INSTANCE_INFO = MODEL_INSTANCE_INFO._replace(name='name')
MODEL_INSTANCE_INFO = MODEL_INSTANCE_INFO._replace(version=0)
MODEL_INSTANCE_INFO = MODEL_INSTANCE_INFO._replace(main_asset_fname='main_asset_fname')
# @launchit.stop

print(MODEL_INSTANCE_INFO._asdict()) # @launchit.collect

{'group_uri': 'group_uri', 'name': 'name', 'version': 0, 'main_asset_fname': 'main_asset_fname'}


In [111]:
ExecGraphEntry = namedtuple('ExecGraphEntry', 'command cell_ind source_line_ind is_oneliner stop_source_line_ind') # @launchit.disable

In [118]:
fname = os.environ['JPY_SESSION_NAME']
fname_dir = os.path.dirname(fname)
fname_name = os.path.splitext(os.path.basename(fname))[0]
fname_ext = os.path.splitext(fname)[1]

new_fname = ''

for i in range(1, 100 + 1):
    new_fname = os.path.join(fname_dir, f'{fname_name}-launch{i}{fname_ext}')

    if not os.path.exists(new_fname):
        break
else:
    raise Exception(f'Failed to generate new launch file name: all variants are taken')

print(f'Creating {new_fname}')

with open(fname, 'r') as f:
    nb = json.load(f)
    exec_graph = []
    collected_source_lines = []

    for cell_ind, cell in enumerate(nb['cells']):
        for source_line_ind, source_line in enumerate(cell['source']):
            source_line = source_line.strip()
            m = re.match(r'^(.*)#\s*@launchit\.(\w+)\s*$', source_line)
            
            if m:
                print(f'Cell {cell_ind}, launchit stanza: "{source_line}"')
                before = m.group(1)
                command = m.group(2)
                ege = ExecGraphEntry(command=None, cell_ind=cell_ind, source_line_ind=source_line_ind, is_oneliner=re.match(r'[^\s]+', before), stop_source_line_ind=-1)

                match command:
                    case 'disable':
                        ege = ege._replace(command=Command.DISABLE)
                    case 'collect':
                        ege = ege._replace(command=Command.COLLECT)
                    case 'collected':
                        assert not ege.is_oneliner, '@launchit.collected cannot be oneliner'
                        ege = ege._replace(command=Command.COLLECTED)
                    case 'stop':
                        assert not ege.is_oneliner, '@launchit.stop cannot be oneliner'
                        # look behind and patch stop_source_line_ind
                        for lb_ege_ind in range(len(exec_graph) - 1, -1, -1):
                            lb_ege = exec_graph[lb_ege_ind]
                            
                            if lb_ege.cell_ind != cell_ind:
                                raise Exception(f'Cell {cell_ind}, line {source_line_ind}, @launchit.stop has no preceeding command')
                            elif lb_ege.stop_source_line_ind == -1:
                                exec_graph[lb_ege_ind] = lb_ege._replace(stop_source_line_ind=source_line_ind)
                                print(f'Cell {cell_ind}, command {lb_ege.command.name} at line {lb_ege.source_line_ind} will stop at line {source_line_ind}')
                                break
                    case _:
                        print(f'WARNING! Cell {cell_ind} contains unrecognized launchit command: "{command}"')

                if not ege.command is None:
                    exec_graph.append(ege)

    for ege in sorted(exec_graph):
        cell = nb['cells'][ege.cell_ind]
        stop_source_line_ind = len(cell['source']) if ege.stop_source_line_ind == -1 else ege.stop_source_line_ind
        
        match ege.command:
            case Command.COLLECT:
                if ege.is_oneliner:
                    print(f'Cell {ege.cell_ind}, collecting source line {ege.source_line_ind}')
                    source_line = cell['source'][ege.source_line_ind]
                    collected_source_lines.append(source_line)
                else:
                    assert ege.source_line_ind + 1 < stop_source_line_ind
                    print(f'Cell {ege.cell_ind}, collecting source lines from {ege.source_line_ind + 1} to {stop_source_line_ind}')

                    if collected_source_lines:
                        collected_source_lines.append('\n')
    
                    for source_line_ind in range(ege.source_line_ind + 1, stop_source_line_ind):
                        source_line = cell['source'][source_line_ind]
                        collected_source_lines.append(source_line)
            case Command.COLLECTED:
                print(f'Cell {ege.cell_ind}, putting {len(collected_source_lines)} collected source lines to {ege.source_line_ind}')

                for ind, source_line in enumerate(collected_source_lines):
                    if ind > 0:
                        cell['source'].insert(ege.source_line_ind + ind, source_line)
                    else:
                        cell['source'][ege.source_line_ind + ind] = source_line
            case Command.DISABLE:
                def disable_source_line(s):
                    if re.match(r'^\s*#', s):
                        return s # already disabled
                    else:
                        return '# ' + s
                
                if ege.is_oneliner:
                    print(f'Cell {ege.cell_ind}, disabling source line {ege.source_line_ind}')
                    cell['source'][ege.source_line_ind] = disable_source_line(cell['source'][ege.source_line_ind])
                else:
                    assert ege.source_line_ind + 1 < stop_source_line_ind
                    print(f'Cell {ege.cell_ind}, disabling source lines from {ege.source_line_ind + 1} to {stop_source_line_ind}')

                    for source_line_ind in range(ege.source_line_ind + 1, len(cell['source'])):
                        cell['source'][source_line_ind] = disable_source_line(cell['source'][source_line_ind])
            case _:
                assert False, f'Failed to understand exec_graph_entry={ege}'

with open(new_fname, 'w') as f:
    json.dump(nb, f, indent=2)

print(f'Created "{new_fname}"')

Creating /home/misha/dev/mine/neurovision/lib/test/launchit_test-launch7.ipynb
Cell 0, launchit stanza: "# @launchit.collected"
Cell 1, launchit stanza: "from collections import namedtuple # @launchit.collect"
Cell 1, launchit stanza: "from enum import IntEnum, auto # @launchit.collect"
Cell 1, launchit stanza: "# @launchit.disable"
Cell 2, launchit stanza: "# @launchit.disable"
Cell 2, launchit stanza: "# @launchit.collect"
Cell 4, launchit stanza: "#     launchit.launchit(os.environ['JPY_SESSION_NAME']) # @launchit.disable"
Cell 5, launchit stanza: "# @launchit.disable"
Cell 5, launchit stanza: "# @launchit.collect"
Cell 5, launchit stanza: "# @launchit.stop"
Cell 5, command COLLECT at line 1 will stop at line 8
Cell 5, launchit stanza: "# @launchit.stop"
Cell 5, command DISABLE at line 0 will stop at line 9
Cell 5, launchit stanza: "# @launchit.disable"
Cell 5, launchit stanza: "# @launchit.stop"
Cell 5, command DISABLE at line 11 will stop at line 16
Cell 5, launchit stanza: "print